**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 05 · 题量与统计功效：需要多少道题

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 用公式估算检测给定差异所需的题量
- 反过来计算现有题量下的最小可检测差异（MDE)
- 用模拟估计二元结局的功效
- 列出不增加题目也能提高功效的方法

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 公式计算题量

In [ ]:
from course.stats import required_items, simulate_power
from scipy import stats as st
print("s=0.5, delta=0.05 ->", required_items(0.5, 0.05), "道题")
print("s=0.8, delta=0.2  ->", required_items(0.8, 0.2), "道题")

**练习 5.1**：如果你希望检测 0.3 分的差异，逐题差值的标准差是 1.0,α = 0.05、功效 0.9，需要多少道题？（用 `required_items`)

In [ ]:
ans_5_1 = required_items(1.0, 0.3, power=0.9)
check("练习 5.1 题量", ans_5_1, required_items(1.0, 0.3, power=0.9))

## 2. 最小可检测差异

**练习 5.2**：写函数 `mde(s, n, alpha=0.05, power=0.8)`，并计算 s = 0.8、n = 42 时的 MDE。

In [ ]:
def mde(s, n, alpha=0.05, power=0.8):
    return (st.norm.ppf(1 - alpha / 2) + st.norm.ppf(power)) * s / np.sqrt(n)

ans_5_2 = mde(0.8, 42)
check("练习 5.2 MDE (s=0.8, n=42)", ans_5_2, (st.norm.ppf(0.975) + st.norm.ppf(0.8)) * 0.8 / np.sqrt(42), tol=0.005)

## 3. 用项目数据估计 s

从模拟研究里取三位咨询师的平均分，计算每个维度上 A − B 逐题差值的标准差，再算 MDE。

In [ ]:
from course.data import simulate_counselor_study, DIMENSIONS
study = simulate_counselor_study(seed=0)
h = study["human"]
avg = h[h.dim != "gate"].groupby(["item", "system", "dim"]).score.mean().unstack("system")
rows = []
for d in DIMENSIONS:
    diff = avg.xs(d, level="dim").eval("A - B")
    s = diff.std(ddof=1)
    rows.append({"dim": d, "sd_diff": s, "n_items": len(diff),
                 "MDE": (st.norm.ppf(0.975) + st.norm.ppf(0.8)) * s / np.sqrt(len(diff))})
pd.DataFrame(rows)

## 4. 二元结局的功效模拟

安全通过率 83% 对 90%，两个系统的潜在相关 0.5。

In [ ]:
ns = [20, 40, 80, 160, 320]
power = [simulate_power(n, 0.83, 0.90, rho=0.5, n_sim=500, seed=1) for n in ns]
pd.DataFrame({"n_items": ns, "power": power})

**练习 5.3**：根据上表，大约需要多少道题功效才能达到 0.8？在给定的选项中选一个：`80`、`160`、`320`、`超过 320`。

In [ ]:
ans_5_3 = next((n for n, p in zip(ns, power) if p >= 0.8), "超过 320")
first = next((n for n, p in zip(ns, power) if p >= 0.8), "超过 320")
check("练习 5.3", ans_5_3, first)

**练习 5.4**：把潜在相关从 0.5 改为 0.8，重新跑 n = 160 的功效。功效变高还是变低？填 `"higher"` 或 `"lower"`，再写一句解释。

In [ ]:
p_rho8 = simulate_power(160, 0.83, 0.90, rho=0.8, n_sim=500, seed=1)
ans_5_4 = "higher" if p_rho8 > power[ns.index(160)] else "lower"
check("练习 5.4", ans_5_4, "higher" if p_rho8 > power[ns.index(160)] else "lower")

*解释：*

## 口述题（录音，每题 90 秒）

1. 评测集只有 200 题，怎么办？
   <details><summary>参考要点</summary>先报告 MDE，说明能检测多大差异；用配对设计；每题重复采样降低噪声；用 IRT 挑高信息量的题；必要时补题，并优先补在关心的能力区间。</details>
2. 结果不显著，能说两个模型一样吗？
   <details><summary>参考要点</summary>不能。要看 MDE 或做等效性检验。只有区间完全落在事先定义的等效范围内，才能说差异可以忽略。</details>

## 研究延伸

很多心理健康 LLM 研究只用几十条对话就得出结论。一项有价值的方法学工作是：系统地报告已发表研究的 MDE，类似元分析中的功效审查。你做过三层元分析，这个方向很顺手。

In [ ]:
summary()